# Risk register and human-oversight checklist
**EU AI Act, Art. 9 (risk management) and Art. 14 (human oversight)**

This notebook turns the results of the CV-screener audit into two documents:
1. a **risk register**: a list of risks, each rated and given a control and an owner;
2. a **human-oversight checklist**: what a human reviewer must be able to do (Art. 14).

How to use it: edit the numbers in section 0, then run all cells from top to bottom.
The numbers are pre-filled from the audit run (gemma3:1b, cut-off 45, 400 CVs).

Note: the screener is high-risk (Annex III 4(a), employment). Its decisions also fall under GDPR Art. 22, so a human must stay in the loop.

## 0. Settings: the audit numbers
Change these after every new audit run. Everything below is computed from them.

In [1]:
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", None)   # show full text in tables
pd.set_option("display.width", 200)

MODEL, CUTOFF = "gemma3:1b", 45

# accuracy and calibration results from the audit
ACCURACY = 0.742
ECE = 0.118            # calibration error (0 = perfect)

# fairness results: gap = score difference, err = its error, sigma = gap / err,
# di = disparate impact (four-fifths rule: below 0.80 is a problem)
FAIRNESS = {
    "age":         {"group": "Candidates aged 52",  "gap":  3.9, "err": 0.8, "sigma":  4.9, "di": 0.83},
    "gender":      {"group": "Female candidates",   "gap": -1.1, "err": 0.5, "sigma": -2.3, "di": 1.11},
    "nationality": {"group": "Non-EU candidates",   "gap":  0.1, "err": 0.5, "sigma":  0.2, "di": 0.99},
}

# tests not run yet: put the number here when you have it (None = not tested)
INJECTION_SUCCESS = None   # share of prompt-injection attacks that worked
DRIFT_ACC_DROP = None      # accuracy lost on new data
PROMPT_SENSITIVE = True    # did a small prompt change change the results?

# targets (illustrative, set your own)
ACC_MIN, ECE_MAX = 0.85, 0.05
REVIEW_BAND = 10           # scores within +-10 of the cut-off go to a human

## 1. How a risk is rated
Every risk gets two numbers from 1 to 3:
- **Likelihood**: how sure are we it happens? (3 = the audit measured it)
- **Impact**: how big is the harm?

**Score = Likelihood x Impact.** Score 6 or more = High, 3 to 4 = Medium, 1 to 2 = Low.

In [2]:
def level(x, low, high):
    # turns a number into 1, 2 or 3: below low = 1, from low = 2, from high = 3
    return 3 if x >= high else 2 if x >= low else 1

def rating(score):
    # turns the score into a word
    return "High" if score >= 6 else "Medium" if score >= 3 else "Low"

risks = []   # we collect the risks in this list

def add_risk(name, article, evidence, consequence, likelihood, impact, control, owner):
    # adds one risk (one row of the register)
    risks.append({
        "Risk": name, "Article": article, "Evidence (audit)": evidence,
        "Business consequence": consequence,
        "Likelihood": likelihood, "Impact": impact,
        "Score": likelihood * impact, "Rating": rating(likelihood * impact),
        "Control": control, "Owner": owner, "Status": "Open"})

## 2. Build the risks
We make one risk per audit finding. Fairness risks are made in a loop, one per protected attribute.

In [3]:
# --- fairness risks (one per attribute) ---
for attr, f in FAIRNESS.items():
    likelihood = level(abs(f["sigma"]), 2, 3)     # more sigma = more sure it is real
    impact = level(abs(f["gap"]), 1, 3)           # bigger gap = bigger harm
    di_text = "passes" if f["di"] >= 0.80 else "FAILS"
    add_risk(
        name=f"Unequal scores by {attr}",
        article="Art. 10(2)(f)-(g)",
        evidence=f"{f['group']}: score gap {f['gap']:+.1f} points ({abs(f['sigma']):.1f} sigma); four-fifths rule {di_text} (DI {f['di']:.2f})",
        consequence="Discrimination claim; unfair rejection of candidates",
        likelihood=likelihood, impact=impact,
        control=f"Remove {attr} cues from CVs before scoring; re-test the gap each release",
        owner="Model owner")

# --- accuracy risk ---
add_risk(
    name="Accuracy below target", article="Art. 15",
    evidence=f"Accuracy {ACCURACY:.3f} vs target {ACC_MIN:.2f}",
    consequence="Good candidates rejected, weak ones shortlisted",
    likelihood=3 if ACCURACY < ACC_MIN else 1,
    impact=level(ACC_MIN - ACCURACY, 0.03, 0.10),
    control="Use a stronger model; keep a human decision on every shortlist",
    owner="Model owner")

# --- calibration risk ---
add_risk(
    name="Confidence not trustworthy", article="Art. 15, Art. 14(4)(b)",
    evidence=f"Calibration error {ECE:.3f} vs limit {ECE_MAX:.2f}",
    consequence="Reviewers trust a wrong score (automation bias)",
    likelihood=3 if ECE > ECE_MAX else 1,
    impact=level(ECE - ECE_MAX, 0.03, 0.10),
    control="Do not show confidence to reviewers until recalibrated",
    owner="Model owner")

# --- prompt sensitivity risk ---
add_risk(
    name="Results change with small prompt changes", article="Art. 15",
    evidence="Audit: fairness and accuracy moved when the prompt was reworded",
    consequence="Unstable decisions; a prompt edit can silently add bias",
    likelihood=3 if PROMPT_SENSITIVE else 1, impact=2,
    control="Freeze the prompt; re-run the audit after any change",
    owner="Model owner")

# --- prompt injection risk (not tested yet -> likelihood 2 = unknown) ---
add_risk(
    name="Prompt injection in CVs", article="Art. 15(5)",
    evidence="Not tested yet" if INJECTION_SUCCESS is None else f"{INJECTION_SUCCESS:.0%} of attacks succeeded",
    consequence="A candidate hides an instruction in the CV and gets shortlisted",
    likelihood=2 if INJECTION_SUCCESS is None else level(INJECTION_SUCCESS, 0.05, 0.20),
    impact=3,
    control="Run the injection test; filter CV text; human check of top-ranked CVs",
    owner="Security lead")

# --- drift risk (not tested yet) ---
add_risk(
    name="Accuracy drops on new job types", article="Art. 72",
    evidence="Not tested yet" if DRIFT_ACC_DROP is None else f"Accuracy dropped by {DRIFT_ACC_DROP:.1%}",
    consequence="Quality falls silently when CVs or roles change",
    likelihood=2 if DRIFT_ACC_DROP is None else level(DRIFT_ACC_DROP, 0.03, 0.08),
    impact=2,
    control="Monthly re-validation on fresh labelled CVs",
    owner="Model owner")

## 3. The risk register
Sorted from highest to lowest score. Add IDs R1, R2, ... so people can refer to a risk.

In [4]:
register = pd.DataFrame(risks)
# sort: highest score first (mergesort keeps the original order for ties)
register = register.sort_values("Score", ascending=False, kind="mergesort").reset_index(drop=True)
register.insert(0, "ID", ["R" + str(i + 1) for i in range(len(register))])

print("Number of risks by rating:")
print(register["Rating"].value_counts().to_string())
display(register[["ID", "Risk", "Article", "Likelihood", "Impact", "Score", "Rating", "Status"]])

Number of risks by rating:
Rating
High      5
Medium    2
Low       1


,ID,Risk,Article,Likelihood,Impact,Score,Rating,Status
0,R1,Unequal scores by age,Art. 10(2)(f)-(g),3,3,9,High,Open
1,R2,Accuracy below target,Art. 15,3,3,9,High,Open
2,R3,Confidence not trustworthy,"Art. 15, Art. 14(4)(b)",3,2,6,High,Open
3,R4,Results change with small prompt changes,Art. 15,3,2,6,High,Open
4,R5,Prompt injection in CVs,Art. 15(5),2,3,6,High,Open
5,R6,Unequal scores by gender,Art. 10(2)(f)-(g),2,2,4,Medium,Open
6,R7,Accuracy drops on new job types,Art. 72,2,2,4,Medium,Open
7,R8,Unequal scores by nationality,Art. 10(2)(f)-(g),1,1,1,Low,Open


In [5]:
# the full text columns (evidence, consequence, control, owner)
display(register[["ID", "Evidence (audit)", "Business consequence", "Control", "Owner"]])

,ID,Evidence (audit),Business consequence,Control,Owner
0,R1,Candidates aged 52: score gap +3.9 points (4.9 sigma); four-fifths rule passes (DI 0.83),Discrimination claim; unfair rejection of candidates,Remove age cues from CVs before scoring; re-test the gap each release,Model owner
1,R2,Accuracy 0.742 vs target 0.85,"Good candidates rejected, weak ones shortlisted",Use a stronger model; keep a human decision on every shortlist,Model owner
2,R3,Calibration error 0.118 vs limit 0.05,Reviewers trust a wrong score (automation bias),Do not show confidence to reviewers until recalibrated,Model owner
3,R4,Audit: fairness and accuracy moved when the prompt was reworded,Unstable decisions; a prompt edit can silently add bias,Freeze the prompt; re-run the audit after any change,Model owner
4,R5,Not tested yet,A candidate hides an instruction in the CV and gets shortlisted,Run the injection test; filter CV text; human check of top-ranked CVs,Security lead
5,R6,Female candidates: score gap -1.1 points (2.3 sigma); four-fifths rule passes (DI 1.11),Discrimination claim; unfair rejection of candidates,Remove gender cues from CVs before scoring; re-test the gap each release,Model owner
6,R7,Not tested yet,Quality falls silently when CVs or roles change,Monthly re-validation on fresh labelled CVs,Model owner
7,R8,Non-EU candidates: score gap +0.1 points (0.2 sigma); four-fifths rule passes (DI 0.99),Discrimination claim; unfair rejection of candidates,Remove nationality cues from CVs before scoring; re-test the gap each release,Model owner


## 4. Human-oversight checklist (Art. 14)
Art. 14(4) says the people overseeing the system must be able to do five things.
We add four more points that follow from Art. 12, 26, 72 and 73.
Status is one of: **Met**, **Partly met**, **To implement**, **Gap**.

In [6]:
# each row: requirement, article, status, what to do
checklist = [
    ("Reviewers understand the model's limits", "Art. 13, 14(4)(a)", "Partly met",
     "Give reviewers a one-page note: what the model can and cannot do (use the audit results)"),
    ("Reviewers are warned about automation bias", "Art. 14(4)(b)",
     "Gap" if ECE > ECE_MAX else "Met",
     "Train reviewers; do not show confidence until it is calibrated"),
    ("Reviewers can interpret the output", "Art. 14(4)(c)", "Partly met",
     "Show the score next to the CV facts it is based on"),
    ("A human can override any decision", "Art. 14(4)(d)", "To implement",
     "Add an 'override' button and record the reason"),
    ("A human can stop the system", "Art. 14(4)(e)", "To implement",
     "Add a stop switch; define who may press it"),
    (f"Borderline cases (score within +-{REVIEW_BAND} of cut-off {CUTOFF}) go to a human",
     "Art. 14, 26(2)", "To implement",
     "Route these CVs to a trained reviewer automatically"),
    ("Every decision is logged and kept at least 6 months", "Art. 12, 26(6)", "Partly met",
     "Log CV id, score, decision, reviewer, time"),
    ("The system is monitored after launch", "Art. 72, 26(5)", "To implement",
     "Monthly re-validation (Project 2)"),
    ("Serious incidents are escalated", "Art. 73, 26(5)", "To implement",
     "Write an escalation path: who to tell, within what time"),
]
oversight = pd.DataFrame(checklist, columns=["Requirement", "Article", "Status", "Action"])
oversight.insert(0, "ID", ["C" + str(i + 1) for i in range(len(oversight))])

print("Checklist status:")
print(oversight["Status"].value_counts().to_string())
display(oversight)

Checklist status:
Status
To implement    5
Partly met      3
Gap             1


,ID,Requirement,Article,Status,Action
0,C1,Reviewers understand the model's limits,"Art. 13, 14(4)(a)",Partly met,Give reviewers a one-page note: what the model can and cannot do (use the audit results)
1,C2,Reviewers are warned about automation bias,Art. 14(4)(b),Gap,Train reviewers; do not show confidence until it is calibrated
2,C3,Reviewers can interpret the output,Art. 14(4)(c),Partly met,Show the score next to the CV facts it is based on
3,C4,A human can override any decision,Art. 14(4)(d),To implement,Add an 'override' button and record the reason
4,C5,A human can stop the system,Art. 14(4)(e),To implement,Add a stop switch; define who may press it
5,C6,Borderline cases (score within +-10 of cut-off 45) go to a human,"Art. 14, 26(2)",To implement,Route these CVs to a trained reviewer automatically
6,C7,Every decision is logged and kept at least 6 months,"Art. 12, 26(6)",Partly met,"Log CV id, score, decision, reviewer, time"
7,C8,The system is monitored after launch,"Art. 72, 26(5)",To implement,Monthly re-validation (Project 2)
8,C9,Serious incidents are escalated,"Art. 73, 26(5)",To implement,"Write an escalation path: who to tell, within what time"


## 5. Save the results
Writes two CSV files and one markdown report that you can put into your portfolio.

In [7]:
def md_table(df):
    # turns a table into markdown text ("|" inside text is escaped)
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(str(v).replace("|", "\\|") for v in row) + " |")
    return "\n".join(lines)

register.to_csv("risk_register.csv", index=False)
oversight.to_csv("oversight_checklist.csv", index=False)

report = (
    "# Risk register and human-oversight checklist\n\n"
    f"Model: {MODEL}, cut-off {CUTOFF}. Basis: Art. 9 and Art. 14, EU AI Act.\n\n"
    "## Risk register\n\n" + md_table(register) + "\n\n"
    "## Human-oversight checklist\n\n" + md_table(oversight) + "\n\n"
    "Scores = likelihood x impact (High 6+, Medium 3-4, Low 1-2). "
    "Thresholds are illustrative. Owners are roles. "
    "This is not legal advice and not an Art. 43 conformity assessment.\n")
with open("risk_register.md", "w", encoding="utf-8") as fh:
    fh.write(report)

print("Saved: risk_register.csv, oversight_checklist.csv, risk_register.md")
display(Markdown(report))

Saved: risk_register.csv, oversight_checklist.csv, risk_register.md


# Risk register and human-oversight checklist

Model: gemma3:1b, cut-off 45. Basis: Art. 9 and Art. 14, EU AI Act.

## Risk register

| ID | Risk | Article | Evidence (audit) | Business consequence | Likelihood | Impact | Score | Rating | Control | Owner | Status |
|---|---|---|---|---|---|---|---|---|---|---|---|
| R1 | Unequal scores by age | Art. 10(2)(f)-(g) | Candidates aged 52: score gap +3.9 points (4.9 sigma); four-fifths rule passes (DI 0.83) | Discrimination claim; unfair rejection of candidates | 3 | 3 | 9 | High | Remove age cues from CVs before scoring; re-test the gap each release | Model owner | Open |
| R2 | Accuracy below target | Art. 15 | Accuracy 0.742 vs target 0.85 | Good candidates rejected, weak ones shortlisted | 3 | 3 | 9 | High | Use a stronger model; keep a human decision on every shortlist | Model owner | Open |
| R3 | Confidence not trustworthy | Art. 15, Art. 14(4)(b) | Calibration error 0.118 vs limit 0.05 | Reviewers trust a wrong score (automation bias) | 3 | 2 | 6 | High | Do not show confidence to reviewers until recalibrated | Model owner | Open |
| R4 | Results change with small prompt changes | Art. 15 | Audit: fairness and accuracy moved when the prompt was reworded | Unstable decisions; a prompt edit can silently add bias | 3 | 2 | 6 | High | Freeze the prompt; re-run the audit after any change | Model owner | Open |
| R5 | Prompt injection in CVs | Art. 15(5) | Not tested yet | A candidate hides an instruction in the CV and gets shortlisted | 2 | 3 | 6 | High | Run the injection test; filter CV text; human check of top-ranked CVs | Security lead | Open |
| R6 | Unequal scores by gender | Art. 10(2)(f)-(g) | Female candidates: score gap -1.1 points (2.3 sigma); four-fifths rule passes (DI 1.11) | Discrimination claim; unfair rejection of candidates | 2 | 2 | 4 | Medium | Remove gender cues from CVs before scoring; re-test the gap each release | Model owner | Open |
| R7 | Accuracy drops on new job types | Art. 72 | Not tested yet | Quality falls silently when CVs or roles change | 2 | 2 | 4 | Medium | Monthly re-validation on fresh labelled CVs | Model owner | Open |
| R8 | Unequal scores by nationality | Art. 10(2)(f)-(g) | Non-EU candidates: score gap +0.1 points (0.2 sigma); four-fifths rule passes (DI 0.99) | Discrimination claim; unfair rejection of candidates | 1 | 1 | 1 | Low | Remove nationality cues from CVs before scoring; re-test the gap each release | Model owner | Open |

## Human-oversight checklist

| ID | Requirement | Article | Status | Action |
|---|---|---|---|---|
| C1 | Reviewers understand the model's limits | Art. 13, 14(4)(a) | Partly met | Give reviewers a one-page note: what the model can and cannot do (use the audit results) |
| C2 | Reviewers are warned about automation bias | Art. 14(4)(b) | Gap | Train reviewers; do not show confidence until it is calibrated |
| C3 | Reviewers can interpret the output | Art. 14(4)(c) | Partly met | Show the score next to the CV facts it is based on |
| C4 | A human can override any decision | Art. 14(4)(d) | To implement | Add an 'override' button and record the reason |
| C5 | A human can stop the system | Art. 14(4)(e) | To implement | Add a stop switch; define who may press it |
| C6 | Borderline cases (score within +-10 of cut-off 45) go to a human | Art. 14, 26(2) | To implement | Route these CVs to a trained reviewer automatically |
| C7 | Every decision is logged and kept at least 6 months | Art. 12, 26(6) | Partly met | Log CV id, score, decision, reviewer, time |
| C8 | The system is monitored after launch | Art. 72, 26(5) | To implement | Monthly re-validation (Project 2) |
| C9 | Serious incidents are escalated | Art. 73, 26(5) | To implement | Write an escalation path: who to tell, within what time |

Scores = likelihood x impact (High 6+, Medium 3-4, Low 1-2). Thresholds are illustrative. Owners are roles. This is not legal advice and not an Art. 43 conformity assessment.


## Notes
- Update section 0 after each audit run and re-run; the register updates itself.
- When you run the injection and drift tests, replace `None` with the result; the likelihood then becomes measured instead of "unknown".
- Thresholds (targets, review band, rating limits) are illustrative: choose values you can defend.
- Owners are roles, not people.